In [1]:
import asyncio

import numpy as np
from assistant.chat_clients import OpenAIChatClient
from assistant.utils import AudioChunk
from dotenv import load_dotenv


load_dotenv()

for i in range(10):
    openai_chat_client = OpenAIChatClient(
        system_prompt="Always reply VERY concisely.",
        voice="ash",
        talking_speed=1.2,
        instant_greeting=False,
    )

    chat_run_task = asyncio.create_task(openai_chat_client.run())

    while True:
        if openai_chat_client._session:
            await openai_chat_client._session.send_message("hello")
            break
        await asyncio.sleep(0.1)

    intro_audio = AudioChunk.empty(sample_rate=24000)

    while True:
        new_chunk = await openai_chat_client.pull_response_audio()
        if isinstance(new_chunk, AudioChunk):
            intro_audio += new_chunk
            print(intro_audio.duration())
        elif new_chunk is None and intro_audio.duration() > 0.1:
            break  # model turn is over
        await asyncio.sleep(0.1)

    np.save(f"src/assistant/greetings/{i + 1}.npy", intro_audio.samples)
    chat_run_task.cancel()

/Users/jckpn-work/dev/assistant-redux/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
3.2602083333333334
3.3460833333333335
0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
3.012125
0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
3.2602083333333334
3.422416666666667
0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
3.2602083333333334
3.422416666666667
0.26266666666666666
0.596625
0.9305833333333333
1.255
1.5889583333333333
1.9229166666666666
2.256875
2.5908333333333333
2.92625
3.2602083333333334
3.5941666666666667
3.6323333333333334
0.26266666666666666
0.596625
0.9305833333333333
1.255


In [4]:
from reachy_mini import ReachyMini
from assistant.audio_transports import ReachyAudioTransport


samples = np.load("src/assistant/greetings/8.npy")
loaded_audio = AudioChunk(samples=samples, sample_rate=24000)

reachy = ReachyMini()
audio_transport = ReachyAudioTransport(reachy)
audio_transport.start()
audio_transport.push_to_speaker(loaded_audio)


GStreamer pipeline error (domain=gst-stream-error-quark, code=1): Internal data stream error.
Dropped 4160 samples. This is most likely because downstream can't keep up and is consuming samples too slowly.
